# Lab 14 — LangGraph supervisor bridge (reference solution)

Canonical LangGraph rebuild of [Lab 10's supervisor-researcher-writer pattern](../../10-supervisor-worker-from-scratch/solution/README.md). Manual supervisor-via-tools, `create_agent` for the researcher, `MessagesState`-extending TypedDict for state.

> 📖 See [`solution/README.md`](./README.md) for the design choices flagged.
> ⏱ Run time: 15-30 seconds end-to-end.


## Setup

Provider-agnostic chat model. Set `PROVIDER` and the corresponding API key env var before running.

In [ ]:
import json
import os
import re
import warnings
from typing import Any, Literal

from dotenv import load_dotenv

# Load .env from repo root if present
import pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = os.environ.get("PROVIDER", "openai")
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=MODEL, temperature=0)

print(f"Provider: {PROVIDER}, model: {MODEL}")


## State schema

`MessagesState`-extending TypedDict. `create_agent` (LangGraph prebuilt) doesn't support Pydantic state, so the framework picks the shape.

In [ ]:
from langgraph.graph import MessagesState


class SupervisorState(MessagesState):
    """Inherits messages: Annotated[list[AnyMessage], add_messages].

    Adds the worker-output fields the supervisor passes between agents.
    """
    findings: str
    citations: list[dict]
    brief_status: str            # "ok" or "step_cap"
    final_answer: str
    last_worker: str             # "researcher" | "writer" — supervisor reads to route


## Web tools

Same web_search + fetch_page tools as Lab 10. Action-hash dedup is built into the tool wrappers; the agent loop doesn't need to know.

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

from langchain_core.tools import tool

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

_RECENCY_MAP = {"any": None, "day": "d", "week": "w", "month": "m", "year": "y"}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")


@tool
def web_search(query: str, recency: str = "any", max_results: int = 8) -> str:
    """Search the web. Returns up to max_results items with title, url, snippet."""
    if not query or not query.strip():
        return json.dumps({"status": "error", "kind": "other", "detail": "empty query"})
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(
                query=query.strip(), region="us-en", safesearch="moderate",
                timelimit=_RECENCY_MAP.get(recency),
                max_results=max_results, backend="auto",
            )
    except (RatelimitException, TimeoutException, DDGSException) as e:
        kind = {"RatelimitException": "rate_limit",
                "TimeoutException": "timeout"}.get(type(e).__name__, "other")
        return json.dumps({"status": "error", "kind": kind, "detail": str(e)})
    except Exception as e:
        return json.dumps({"status": "error", "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})
    if not raw:
        return json.dumps({"status": "empty", "query": query, "detail": "no results"})
    return json.dumps({
        "status": "ok",
        "results": [{"title": (r.get("title") or "").strip(),
                     "url": (r.get("href") or "").strip(),
                     "snippet": (r.get("body") or "").strip()}
                    for r in raw if r.get("href")][:max_results],
    })


@tool
def fetch_page(url: str, max_chars: int = 8000) -> str:
    """Fetch the full text content of a URL."""
    if not url or not url.startswith(("http://", "https://")):
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": "invalid url"})
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15)
        resp.raise_for_status()
    except requests.HTTPError:
        kind = "http_5xx" if 500 <= resp.status_code < 600 else "http_4xx"
        return json.dumps({"status": "error", "url": url, "kind": kind,
                           "detail": f"HTTP {resp.status_code}"})
    except requests.RequestException as e:
        return json.dumps({"status": "error", "url": url, "kind": "other",
                           "detail": f"{type(e).__name__}: {e}"})

    soup = BeautifulSoup(resp.text, "html.parser")
    for el in soup(["script", "style", "nav", "footer", "aside"]):
        el.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(separator=" ")).strip()[:max_chars]
    title = (soup.title.string or "").strip() if soup.title else ""
    return json.dumps({"status": "ok", "url": url, "title": title, "text": text})


## Researcher worker — `create_agent`

The framework provides the agent loop. The researcher's job is unchanged from Lab 10: search, fetch, return a JSON envelope.

In [ ]:
from langchain.agents import create_agent

RESEARCHER_SYSTEM_PROMPT = """You are a research worker. Given a question:
1. Use web_search to find relevant pages.
2. Use fetch_page on the most promising 2-3 URLs to read full content.
3. Synthesize findings into 100-200 words, with inline citation markers [1], [2], etc.
4. Stop after at most 6 tool calls total.

Output ONLY a JSON object (no markdown fences) with this shape:
{
  "findings": "<150 words of synthesized prose with [1] [2] citation markers>",
  "citations": [
    {"url": "...", "title": "..."},
    ...
  ]
}

The citations list must match the [N] markers in findings exactly."""

researcher = create_agent(llm, tools=[web_search, fetch_page],
                           system_prompt=RESEARCHER_SYSTEM_PROMPT)


## Writer worker

Prompt-only. No tools needed. Reads findings + citations from state, composes prose.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing
findings and a list of citations. Produce ~150 words of clean prose that:

1. Preserves every citation marker [1], [2], ... from findings exactly as given.
2. Lists each citation at the end in the form:
   [N] <title> — <url>
3. Does not introduce claims beyond what findings supports.
4. Does not paraphrase the brief to the point of dropping facts.

If brief_status is "step_cap" or partial, surface that the answer is partial
and proceed with whatever was provided.

Output only the prose; no preamble, no JSON wrapper."""


def writer_node(state: SupervisorState) -> dict:
    """Compose the final answer from findings + citations."""
    findings = state.get("findings", "")
    citations = state.get("citations", [])
    brief_status = state.get("brief_status", "ok")

    brief = (
        f"FINDINGS:\n{findings}\n\n"
        f"CITATIONS:\n{json.dumps(citations, indent=2)}\n\n"
        f"BRIEF_STATUS: {brief_status}\n"
    )
    response = llm.invoke([
        SystemMessage(content=WRITER_SYSTEM_PROMPT),
        HumanMessage(content=brief),
    ])
    return {
        "final_answer": response.content,
        "last_worker": "writer",
        "messages": [AIMessage(content=f"writer_complete: {response.content[:120]}...",
                                name="writer")],
    }


## Researcher node wrapper

Translates state → sub-agent invocation, parses the JSON envelope back into state fields.

In [ ]:
def researcher_node(state: SupervisorState) -> dict:
    """Invoke the researcher sub-agent. Parse its JSON envelope into state fields."""
    last_user = next(
        (m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)),
        None,
    )
    if last_user is None:
        return {"findings": "", "citations": [], "brief_status": "error",
                "last_worker": "researcher"}

    result = researcher.invoke({"messages": [last_user]})
    raw = result["messages"][-1].content
    if not isinstance(raw, str):
        raw = str(raw)
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()

    try:
        envelope = json.loads(raw)
        return {
            "findings": envelope.get("findings", ""),
            "citations": envelope.get("citations", []),
            "brief_status": "ok",
            "last_worker": "researcher",
            "messages": [AIMessage(content="researcher_complete", name="researcher")],
        }
    except json.JSONDecodeError:
        # Surface as step_cap so the supervisor routes to writer with partial result
        return {
            "findings": raw[:1000],
            "citations": [],
            "brief_status": "step_cap",
            "last_worker": "researcher",
            "messages": [AIMessage(content="researcher_complete (parse failed)",
                                    name="researcher")],
        }


## Supervisor node — manual, not `create_supervisor()`

Routing tools (`call_researcher`, `call_writer`) are stubs — they exist so `bind_tools()` knows their schema. The supervisor node *intercepts* the LLM's tool call to decide where to route via `Command(goto=...)`.

In [ ]:
from langgraph.types import Command


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent coordinating two workers
via tool calls.

WORKFLOW:
1. Call the researcher with the user's question.
2. After the researcher returns, call the writer with the brief.
3. After the writer returns, finalize by responding directly (no tool call).

WORKERS:
- researcher: takes a question, returns findings + citations. Status is "ok"
  or "step_cap".
- writer: takes findings + citations, returns 150 words of cited prose.

RULES:
- Call each worker at most ONCE per task.
- Pass citations VERBATIM from researcher to writer — do not paraphrase.
- If the researcher returns step_cap, still call the writer with the partial brief.

Respond with a tool call to call_researcher or call_writer, or with a final
direct response when the writer has completed.
"""


@tool
def call_researcher(question: str) -> str:
    """Dispatch the question to the researcher worker."""
    return "[routed to researcher node]"


@tool
def call_writer() -> str:
    """Dispatch the brief to the writer worker. Uses findings + citations from state."""
    return "[routed to writer node]"


supervisor_llm = llm.bind_tools([call_researcher, call_writer])


def supervisor_node(
    state: SupervisorState,
) -> Command[Literal["researcher", "writer", "__end__"]]:
    """Routing logic. Returns Command(goto=..., update=...)."""
    response = supervisor_llm.invoke([
        SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT),
        *state["messages"],
    ])
    state_update: dict[str, Any] = {"messages": [response]}

    if not response.tool_calls:
        return Command(goto="__end__", update=state_update)

    tc = response.tool_calls[0]
    if tc["name"] == "call_researcher":
        return Command(goto="researcher", update=state_update)
    elif tc["name"] == "call_writer":
        return Command(goto="writer", update=state_update)
    else:
        return Command(goto="__end__", update=state_update)


## Wire the graph

`StateGraph` + nodes + edges + checkpointer. The `compile()` call returns the runnable graph.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


def build_supervisor_graph(checkpointer=None):
    """Build and compile the supervisor graph."""
    builder = StateGraph(SupervisorState)
    builder.add_node("supervisor", supervisor_node)
    builder.add_node("researcher", researcher_node)
    builder.add_node("writer", writer_node)

    builder.add_edge(START, "supervisor")
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("writer", "supervisor")

    return builder.compile(checkpointer=checkpointer)


graph = build_supervisor_graph(checkpointer=InMemorySaver())
print("Graph compiled.")


## End-to-end run

One demonstration on a research-and-write task. `recursion_limit=12` caps the full graph (supervisor + researcher's internal loop + writer).

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary."
)
config = {"configurable": {"thread_id": "demo"}, "recursion_limit": 12}

result = graph.invoke({"messages": [HumanMessage(content=task)]}, config=config)

print("Final answer:")
print("─" * 60)
print(result.get("final_answer", "[no final answer]"))


**Sample output**:

```
Final answer:
────────────────────────────────────────────────────────────
The Model Context Protocol (MCP) is an open standard for connecting
LLMs to external tools and data sources [1]. Recent developments
include expanded server libraries [2] and broader adoption in
production deployments [3]. ...

[1] Model Context Protocol Overview — https://example-research.org/...
[2] MCP Production Deployments in 2026 — https://example-blog.com/...
[3] MCP Server Gallery — https://example-docs.io/...
```

The full graph runs supervisor → researcher (with its internal `create_agent` loop running web_search and fetch_page) → supervisor → writer → supervisor (END). Six node visits at the graph level; up to ~6 LLM calls total.

## Synthesis

What the framework absorbed:

- The supervisor's loop bookkeeping — `recursion_limit` replaces Lab 10's manual `SUPERVISOR_MAX_STEPS` counter.
- The researcher's agent loop — `create_agent` replaces Lab 10's `chat_with_tools` driver.
- Crash-resume — `InMemorySaver` (or persistent checkpointers) wraps the graph; same routing logic.
- Streaming — `graph.stream(stream_mode="updates")` instead of `invoke` (one-line change).

What stayed unchanged from Lab 10:

- Prompt content (researcher, writer).
- Citation-preservation discipline (writer reads citations from state, doesn't paraphrase).
- Worker contracts (researcher emits JSON envelope; writer takes findings+citations).
- Web tools (web_search, fetch_page) with action-hash dedup at the tool layer.

What carries over to [Lab 15](../../15-langgraph-plan-execute-bridge/solution/README.md): same `StateGraph`/`Command` pattern, different topology (planner + dispatcher + parallel executors + synthesizer instead of supervisor + workers).
